# <center style="font-weight: bold; color: #0098cd;">Transcripción automática del habla (ASR)</center>

## 1. Introducción

### 1.1 Objetivo

### 1.2 Contexto dentro del sistema completo

### 1.3 Requisitos de esta fase

## 2. Preparación del entorno de trabajo

### 2.1 Instalación de dependencias

In [ ]:
# ============================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================
# Este notebook requiere la instalación previa de las dependencias del proyecto.
# Ejecutar en terminal:
# pip install -r requirements.txt

### 2.2 Importación de librerías

# PASAR ESTE CODIGO AL ESTILO DEL NOTEBOOK 1

In [ ]:
# Gestión de rutas y sistema de archivos
from pathlib import Path
import soundfile as sf
import os
import json

# Manipulación de datos
import pandas as pd

# ASR (Whisper)
from faster_whisper import WhisperModel

# Evaluación
from jiwer import wer, cer, process_words

# Utilidades
from tqdm import tqdm

from collections import Counter

import re

### 2.3 Gestión y configuración de rutas

In [4]:
# Detección automática de la raíz del proyecto
project_root = Path.cwd()

while not (project_root / "data").exists():
    project_root = project_root.parent

# Directorio principal de datos
data_dir = project_root / "data"

# =========================
# AUDIOS DE ENTRADA
# =========================
standardized_audio_dir = data_dir / "standardized_audio"   # baseline (sin procesado adicional)

processed_audio_dir = data_dir / "processed_audio"

audio_without_vad_dir = processed_audio_dir / "without_vad"
audio_conditional_vad_dir = processed_audio_dir / "conditional_vad"

# =========================
# TRANSCRIPCIONES
# =========================
transcriptions_dir = data_dir / "transcriptions"

# Predicciones del ASR
predictions_dir = transcriptions_dir / "predictions"

pred_standardized_dir = predictions_dir / "standardized"   # baseline
pred_without_vad_dir = predictions_dir / "without_vad"
pred_conditional_vad_dir = predictions_dir / "conditional_vad"

# Ground truth (transcripciones manuales)
ground_truth_dir = transcriptions_dir / "ground_truth"

# =========================
# CREACIÓN DE DIRECTORIOS
# =========================
pred_standardized_dir.mkdir(parents=True, exist_ok=True)
pred_without_vad_dir.mkdir(parents=True, exist_ok=True)
pred_conditional_vad_dir.mkdir(parents=True, exist_ok=True)
ground_truth_dir.mkdir(parents=True, exist_ok=True)

# =========================
# VERIFICACIÓN
# =========================
print("Ruta raíz:", project_root)

print("\n--- Audios ---")
print("Baseline (standardized):", standardized_audio_dir)
print("Sin VAD:", audio_without_vad_dir)
print("VAD condicional:", audio_conditional_vad_dir)

print("\n--- Transcripciones ---")
print("Predicciones baseline:", pred_standardized_dir)
print("Predicciones sin VAD:", pred_without_vad_dir)
print("Predicciones VAD condicional:", pred_conditional_vad_dir)
print("Ground truth:", ground_truth_dir)

Ruta raíz: /Volumes/EXTENSION/GitHub/TFM

--- Audios ---
Baseline (standardized): /Volumes/EXTENSION/GitHub/TFM/data/standardized_audio
Sin VAD: /Volumes/EXTENSION/GitHub/TFM/data/processed_audio/without_vad
VAD condicional: /Volumes/EXTENSION/GitHub/TFM/data/processed_audio/conditional_vad

--- Transcripciones ---
Predicciones baseline: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/standardized
Predicciones sin VAD: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/without_vad
Predicciones VAD condicional: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/conditional_vad
Ground truth: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/ground_truth


## 3. Preparación del *dataset* para ASR

### 3.1 Carga de audios preprocesados

In [5]:
# Carga de rutas de audios baseline (solo estandarizados)
audio_paths_standardized = sorted(list(standardized_audio_dir.glob("*.wav")))

# Carga de rutas de audios sin VAD
audio_paths_without_vad = sorted(list(audio_without_vad_dir.glob("*.wav")))

# Carga de rutas de audios con VAD condicional
audio_paths_conditional_vad = sorted(list(audio_conditional_vad_dir.glob("*.wav")))

print(f"Audios baseline (standardized): {len(audio_paths_standardized)}")
print(f"Audios sin VAD: {len(audio_paths_without_vad)}")
print(f"Audios con VAD condicional: {len(audio_paths_conditional_vad)}")

Audios baseline (standardized): 11
Audios sin VAD: 11
Audios con VAD condicional: 11


### 3.2 Asociación con metadatos

In [8]:
# Construcción de dataset baseline (solo estandarizado)
dataset_standardized = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_standardized
]

# Construcción de dataset sin VAD
dataset_without_vad = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_without_vad
]

# Construcción de dataset con VAD condicional
dataset_conditional_vad = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_conditional_vad
]

# Conversión a DataFrame para inspección
df_standardized = pd.DataFrame(dataset_standardized)
df_without_vad = pd.DataFrame(dataset_without_vad)
df_conditional_vad = pd.DataFrame(dataset_conditional_vad)

pd.DataFrame({
    "standardized": df_standardized["audio_id"].head(),
    "without_vad": df_without_vad["audio_id"].head(),
    "conditional_vad": df_conditional_vad["audio_id"].head()
})

,standardized,without_vad,conditional_vad
0,AUDIO-2025-09-18-13-34-03,AUDIO-2025-09-18-13-34-03,AUDIO-2025-09-18-13-34-03
1,AUDIO-2026-03-12-09-54-36,AUDIO-2026-03-12-09-54-36,AUDIO-2026-03-12-09-54-36
2,AUDIO-2026-04-21-17-26-57,AUDIO-2026-04-21-17-26-57,AUDIO-2026-04-21-17-26-57
3,AUDIO-2026-04-22-11-23-01,AUDIO-2026-04-22-11-23-01,AUDIO-2026-04-22-11-23-01
4,AUDIO-2026-04-22-11-24-06,AUDIO-2026-04-22-11-24-06,AUDIO-2026-04-22-11-24-06


### 3.3 Validación de entradas para transcripción

#### 3.3.1 Verificación de formato (wav, 16kHz, mono)

In [9]:
# Validación baseline (standardized)
for item in dataset_standardized:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)


# Validación sin VAD
for item in dataset_without_vad:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)


# Validación con VAD condicional
for item in dataset_conditional_vad:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)

#### 3.3.2 Control de errores y audios inválidos

In [11]:
# Filtrado baseline (standardized)
valid_standardized = [item for item in dataset_standardized if item["format_ok"]]
invalid_standardized = [item for item in dataset_standardized if not item["format_ok"]]

print(f"Standardized → válidos: {len(valid_standardized)}, inválidos: {len(invalid_standardized)}")


# Filtrado sin VAD
valid_without_vad = [item for item in dataset_without_vad if item["format_ok"]]
invalid_without_vad = [item for item in dataset_without_vad if not item["format_ok"]]

print(f"Sin VAD → válidos: {len(valid_without_vad)}, inválidos: {len(invalid_without_vad)}")


# Filtrado con VAD condicional
valid_conditional_vad = [item for item in dataset_conditional_vad if item["format_ok"]]
invalid_conditional_vad = [item for item in dataset_conditional_vad if not item["format_ok"]]

print(f"VAD condicional → válidos: {len(valid_conditional_vad)}, inválidos: {len(invalid_conditional_vad)}")


# Mostrar errores si existen
if any([invalid_standardized, invalid_without_vad, invalid_conditional_vad]):
    print("\nEjemplos de errores:")
    
    for item in (invalid_standardized + invalid_without_vad + invalid_conditional_vad)[:5]:
        print(f"- {item['audio_id']}: {item.get('error', 'Formato incorrecto')}")

Standardized → válidos: 11, inválidos: 0
Sin VAD → válidos: 11, inválidos: 0
VAD condicional → válidos: 11, inválidos: 0


## 4. Configuración del modelo ASR

### 4.1 Selección del modelo base (Whisper)

In [12]:
# Selección del tamaño del modelo
model_size = "medium"  # opciones: tiny, base, small, medium, large-v3

# Configuración de dispositivo (ajusta según tu equipo)
device = "cpu"   # "cuda" si tienes GPU
compute_type = "int8"  # "float16" en GPU, "int8" en CPU

# Carga del modelo
model = WhisperModel(model_size, device=device, compute_type=compute_type)

print(f"Modelo cargado: {model_size}")

Modelo cargado: medium


### 4.2 Parámetros de inferencia

In [13]:
asr_params = {
    "language": "es",                       # fuerza el idioma español, evitando autodetección y posibles errores de idioma
    "beam_size": 1,                         # número de hipótesis evaluadas; 1 = decodificación greedy (rápida y casi determinista)
    "condition_on_previous_text": False,    # evita usar contexto previo entre segmentos, reduciendo propagación de errores
    "word_timestamps": False,               # desactiva timestamps por palabra (innecesario si solo quieres texto)
    "temperature": 0.0                      # desactiva aleatoriedad para resultados consistentes
}

### 4.3 Definición de configuración *baseline*

In [14]:
# Configuración baseline utilizada en la transcripción

baseline_config = {
    "model_size": model_size,
    "device": device,
    "compute_type": compute_type,
    "asr_params": asr_params
}

print("Configuración baseline:")
display(baseline_config)

Configuración baseline:


{'model_size': 'medium',
 'device': 'cpu',
 'compute_type': 'int8',
 'asr_params': {'language': 'es',
  'beam_size': 1,
  'condition_on_previous_text': False,
  'word_timestamps': False,
  'temperature': 0.0}}

## 5. Transcripción automática del audio

INTRO AQUI

### 5.1 Generación de transcripciones (*baseline*)

In [15]:
# Transcripción baseline (standardized)
results_standardized = []

for item in tqdm(dataset_standardized):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_standardized.append({
        "audio_id": item["audio_id"],
        "text": text
    })


# Transcripción sin VAD
results_without_vad = []

for item in tqdm(dataset_without_vad):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_without_vad.append({
        "audio_id": item["audio_id"],
        "text": text
    })


# Transcripción con VAD condicional
results_conditional_vad = []

for item in tqdm(dataset_conditional_vad):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_conditional_vad.append({
        "audio_id": item["audio_id"],
        "text": text
    })

100%|██████████| 11/11 [03:16<00:00, 17.90s/it]


### 5.2 Almacenamiento estructurado de resultados

Antes de ejecutar este bloque, se recomienda limpiar manualmente los directorios de salida para evitar inconsistencias entre ejecuciones.

In [16]:
# Guardado baseline (standardized)
for item in results_standardized:
    output_path = pred_standardized_dir / f"{item['audio_id']}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)


# Guardado sin VAD
for item in results_without_vad:
    output_path = pred_without_vad_dir / f"{item['audio_id']}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)


# Guardado con VAD condicional
for item in results_conditional_vad:
    output_path = pred_conditional_vad_dir / f"{item['audio_id']}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)

Las transcripciones generadas se almacenan en formato JSON, donde cada archivo representa un audio procesado. La estructura contiene el identificador del audio (`audio_id`) y el texto transcrito (`text`), facilitando su posterior uso en el cálculo de métricas y en las etapas de procesamiento del lenguaje natural.

Ejemplo:

```json
{
  "audio_id": "001",
  "text": "aplique fungicida en lote 3"
}
```

### 5.3 Verificación de transcripciones generadas

In [19]:
# Ejemplo de verificación
print(results_standardized[0])
print(results_without_vad[0])
print(results_conditional_vad[0])

{'audio_id': 'AUDIO-2025-09-18-13-34-03', 'text': 'Buenas, José Manuel.  Mándame porfa si puedes algún teléfono de aquí de Ciuri o del encargado o del alcalde  o de alguien de aquí porfa.'}
{'audio_id': 'AUDIO-2025-09-18-13-34-03', 'text': 'Buenas José Manuel, mandame porfa si puedes algún teléfono de aquí de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ci'}
{'audio_id': 'AUDIO-2025-09-18-13-34-03', 'text': 'Buenas José Manuel, mandame porfa si puedes algún teléfono de aquí de Ciudad de Ciudad de Ciudad de Ciudad de Ciudad de Ciu

### 5.4 Control de errores

In [20]:
# Detección de textos vacíos
empty_standardized = [r for r in results_standardized if r["text"] == ""]
empty_without_vad = [r for r in results_without_vad if r["text"] == ""]
empty_conditional_vad = [r for r in results_conditional_vad if r["text"] == ""]

print(f"Vacíos baseline: {len(empty_standardized)}")
print(f"Vacíos sin VAD: {len(empty_without_vad)}")
print(f"Vacíos con VAD condicional: {len(empty_conditional_vad)}")

Vacíos baseline: 0
Vacíos sin VAD: 0
Vacíos con VAD condicional: 0


## 6. Evaluación del rendimiento del ASR

### 6.1 Definición del conjunto de referencia (*ground truth*)

subconjunto etiquetado manualmente

In [ ]:
# Carga de transcripciones manuales (ground truth)
import json

ground_truth = {}

for path in ground_truth_dir.glob("*.json"):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        ground_truth[data["audio_id"]] = data["text"]

print(f"Total ground truth: {len(ground_truth)}")

### 6.2 Cálculo de métricas

INTRO AQUI

#### 6.2.1 *Word Error Rate* (WER)

In [ ]:
# Función para calcular el Word Error Rate (WER) para un conjunto de resultados
def compute_wer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de WER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Cálculo del WER entre referencia y predicción
            score = wer(ref, hyp)
            
            # Almacenamiento del resultado individual
            scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

#### 6.2.2 *Character Error Rate* (CER)

In [ ]:
# Función para calcular el Character Error Rate (CER) para un conjunto de resultados
def compute_cer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de CER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Cálculo del CER entre referencia y predicción
            score = cer(ref, hyp)
            
            # Almacenamiento del resultado individual
            scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

### 6.3 Evaluación cuantitativa del *baseline*

In [ ]:
# =========================
# CÁLCULO DE MÉTRICAS
# =========================

# Cálculo del Word Error Rate (WER) para cada pipeline
wer_standardized = compute_wer(results_standardized, ground_truth)
wer_without_vad = compute_wer(results_without_vad, ground_truth)
wer_conditional_vad = compute_wer(results_conditional_vad, ground_truth)

# Cálculo del Character Error Rate (CER) para cada pipeline
cer_standardized = compute_cer(results_standardized, ground_truth)
cer_without_vad = compute_cer(results_without_vad, ground_truth)
cer_conditional_vad = compute_cer(results_conditional_vad, ground_truth)


# =========================
# CÁLCULO DE VALORES MEDIOS
# =========================

# Cálculo del WER medio para cada enfoque
mean_wer_standardized = sum(wer_standardized) / len(wer_standardized)
mean_wer_without_vad = sum(wer_without_vad) / len(wer_without_vad)
mean_wer_conditional_vad = sum(wer_conditional_vad) / len(wer_conditional_vad)

# Cálculo del CER medio para cada enfoque
mean_cer_standardized = sum(cer_standardized) / len(cer_standardized)
mean_cer_without_vad = sum(cer_without_vad) / len(cer_without_vad)
mean_cer_conditional_vad = sum(cer_conditional_vad) / len(cer_conditional_vad)


# =========================
# VISUALIZACIÓN DE RESULTADOS
# =========================

print("WER medio:")
print(f"Standardized: {mean_wer_standardized:.4f}")
print(f"Sin VAD: {mean_wer_without_vad:.4f}")
print(f"VAD condicional: {mean_wer_conditional_vad:.4f}")

print("\nCER medio:")
print(f"Standardized: {mean_cer_standardized:.4f}")
print(f"Sin VAD: {mean_cer_without_vad:.4f}")
print(f"VAD condicional: {mean_cer_conditional_vad:.4f}")

### 6.4 Análisis de resultados globales

In [ ]:
# Comprobamos que los audio_id coinciden en los 3 pipelines
ids_std = [r["audio_id"] for r in results_standardized]
ids_no_vad = [r["audio_id"] for r in results_without_vad]
ids_vad = [r["audio_id"] for r in results_conditional_vad]

assert ids_std == ids_no_vad == ids_vad, "Error: los audio_id no están alineados"

# Construcción de DataFrame con resultados individuales por audio
df_results = pd.DataFrame({
    "audio_id": ids_std,
    "wer_standardized": wer_standardized,
    "wer_without_vad": wer_without_vad,
    "wer_conditional_vad": wer_conditional_vad,
    "cer_standardized": cer_standardized,
    "cer_without_vad": cer_without_vad,
    "cer_conditional_vad": cer_conditional_vad
})

# Mejor pipeline por WER y CER
df_results["best_wer"] = df_results[
    ["wer_standardized", "wer_without_vad", "wer_conditional_vad"]
].idxmin(axis=1)

df_results["best_cer"] = df_results[
    ["cer_standardized", "cer_without_vad", "cer_conditional_vad"]
].idxmin(axis=1)

# Visualización de los primeros resultados
display(df_results.head())

## 7. Análisis de errores de transcripción

### 7.1 Identificación de errores frecuentes

In [ ]:
# Función para obtener las alineaciones entre referencia y predicción
def get_alignment(results, ground_truth):
    
    # Lista donde se almacenan todas las operaciones de alineación
    alignments = []
    
    # Iteramos sobre cada resultado del ASR
    for item in results:
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo
            hyp = item["text"]
            
            # Cálculo de alineación palabra a palabra
            alignment = process_words(ref, hyp)
            
            # Acumulamos las operaciones (sustitución, inserción, eliminación)
            alignments.extend(alignment.alignments)
    
    return alignments

#### 7.1.1 Sustituciones

In [ ]:
# Función para obtener las sustituciones más frecuentes
def get_substitutions(results, ground_truth):
    
    # Contador de sustituciones (ref → hyp)
    subs = Counter()
    
    # Obtenemos todas las alineaciones
    alignments = get_alignment(results, ground_truth)
    
    # Iteramos sobre cada operación
    for op, ref_word, hyp_word in alignments:
        
        # Filtramos solo sustituciones
        if op == "substitute":
            subs[(ref_word, hyp_word)] += 1
    
    return subs


# Cálculo para los tres pipelines
subs_std = get_substitutions(results_standardized, ground_truth)
subs_no_vad = get_substitutions(results_without_vad, ground_truth)
subs_vad = get_substitutions(results_conditional_vad, ground_truth)

# Visualización
print("Sustituciones más frecuentes:")
print("Standardized:", subs_std.most_common(5))
print("Sin VAD:", subs_no_vad.most_common(5))
print("VAD condicional:", subs_vad.most_common(5))

#### 7.1.2 Inserciones

In [ ]:
# Función para calcular las inserciones más frecuentes
def get_insertions(results, ground_truth):
    
    # Contador de palabras insertadas por el modelo ASR
    ins = Counter()
    
    # Obtención de alineaciones
    alignments = get_alignment(results, ground_truth)
    
    # Iteración sobre operaciones
    for op, _, hyp_word in alignments:
        
        # Filtramos inserciones
        if op == "insert":
            ins[hyp_word] += 1
    
    return ins


# Cálculo para los tres pipelines
ins_std = get_insertions(results_standardized, ground_truth)
ins_no_vad = get_insertions(results_without_vad, ground_truth)
ins_vad = get_insertions(results_conditional_vad, ground_truth)

# Visualización
print("Inserciones más frecuentes:")
print("Standardized:", ins_std.most_common(5))
print("Sin VAD:", ins_no_vad.most_common(5))
print("VAD condicional:", ins_vad.most_common(5))

#### 7.1.3 Eliminaciones

In [ ]:
# Función para calcular las eliminaciones más frecuentes
def get_deletions(results, ground_truth):
    
    # Contador de palabras eliminadas por el modelo ASR
    dels = Counter()
    
    # Obtención de alineaciones
    alignments = get_alignment(results, ground_truth)
    
    # Iteración sobre operaciones
    for op, ref_word, _ in alignments:
        
        # Filtramos eliminaciones
        if op == "delete":
            dels[ref_word] += 1
    
    return dels


# Cálculo para los tres pipelines
dels_std = get_deletions(results_standardized, ground_truth)
dels_no_vad = get_deletions(results_without_vad, ground_truth)
dels_vad = get_deletions(results_conditional_vad, ground_truth)

# Visualización
print("Eliminaciones más frecuentes:")
print("Standardized:", dels_std.most_common(5))
print("Sin VAD:", dels_no_vad.most_common(5))
print("VAD condicional:", dels_vad.most_common(5))

#### 7.1.4 Resumen cuantitativo de errores

In [ ]:
# Construcción de tabla resumen
df_error_summary = pd.DataFrame({
    "Pipeline": ["Standardized", "Sin VAD", "VAD condicional"],
    "Sustituciones": [
        sum(subs_std.values()),
        sum(subs_no_vad.values()),
        sum(subs_vad.values())
    ],
    "Inserciones": [
        sum(ins_std.values()),
        sum(ins_no_vad.values()),
        sum(ins_vad.values())
    ],
    "Eliminaciones": [
        sum(dels_std.values()),
        sum(dels_no_vad.values()),
        sum(dels_vad.values())
    ]
})

# Cálculo del total de errores
df_error_summary["Total"] = (
    df_error_summary["Sustituciones"] +
    df_error_summary["Inserciones"] +
    df_error_summary["Eliminaciones"]
)

display(df_error_summary)

#### 7.1.5 Conclusiones

TEXTO AQUI

### 7.2 Análisis de errores con impacto en la estructuración

#### 7.2.1 Términos de dominio

## ✔ Ejemplo de JSON (lo que deberían darte)

```json
{
  "cultivo_cafe": ["café", "cafetal"],
  "plaga": ["plaga", "bicho", "insecto"],
  "producto_fitosanitario": ["fungicida", "herbicida", "insecticida", "veneno"],
  "unidad_parcela": ["lote", "parcela", "finca"],
  "cosecha": ["cosecha", "recolección"]
}
```

## ✔ Cómo lo justificas en el TFM

> Los términos de dominio son definidos por expertos y almacenados en un archivo externo en formato JSON, lo que permite su actualización independiente del código y facilita su reutilización en etapas posteriores del pipeline.

In [ ]:
# Definición de términos de dominio relevantes para el análisis
domain_terms = ["café", "plaga", "fungicida", "lote", "cosecha"]

# Función para calcular errores en términos de dominio
def domain_error_count(results, ground_truth):
    count = 0

    for item in results:
        audio_id = item["audio_id"]

        # Verificamos que exista referencia en ground truth
        if audio_id in ground_truth:

            # Normalizamos texto a minúsculas para comparación robusta
            ref = ground_truth[audio_id].lower()
            hyp = item["text"].lower()

            # Comprobamos pérdida de términos de dominio
            for term in domain_terms:
                if term in ref and term not in hyp:
                    count += 1

    return count

#### 7.2.2 Nombres propios

In [ ]:
# Función para extraer palabras capitalizadas (aproximación a nombres propios)
def extract_names(text):
    return re.findall(r"\b[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+\b", text)

# Función para calcular errores en nombres propios
def name_error_count(results, ground_truth):
    count = 0

    for item in results:
        audio_id = item["audio_id"]

        # Verificamos existencia en ground truth
        if audio_id in ground_truth:

            # Extracción de nombres en referencia y predicción
            ref_names = extract_names(ground_truth[audio_id])
            hyp_names = extract_names(item["text"])

            # Contabilizamos nombres perdidos
            for name in ref_names:
                if name not in hyp_names:
                    count += 1

    return count

#### 7.2.3 Números y cantidades

In [ ]:
# Función para extraer números en formato dígito y texto
def extract_numbers(text):
    return re.findall(
        r"\d+[.,]?\d*|\b(cero|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|"
        r"diez|once|doce|trece|catorce|quince|veinte|treinta|cuarenta|"
        r"cincuenta|sesenta|setenta|ochenta|noventa|cien|ciento|mil)\b",
        text.lower()
    )

# Función para calcular errores en números
def number_error_count(results, ground_truth):
    count = 0

    for item in results:
        audio_id = item["audio_id"]

        # Verificamos existencia en ground truth
        if audio_id in ground_truth:

            # Extracción de números
            ref_nums = extract_numbers(ground_truth[audio_id])
            hyp_nums = extract_numbers(item["text"])

            # Comprobamos discrepancias
            if ref_nums != hyp_nums:
                count += 1

    return count

#### 7.2.4 Resumen cuantitativo del impacto

In [ ]:
# Cálculo de errores para cada pipeline
domain_std = domain_error_count(results_standardized, ground_truth)
domain_no_vad = domain_error_count(results_without_vad, ground_truth)
domain_vad = domain_error_count(results_conditional_vad, ground_truth)

name_std = name_error_count(results_standardized, ground_truth)
name_no_vad = name_error_count(results_without_vad, ground_truth)
name_vad = name_error_count(results_conditional_vad, ground_truth)

number_std = number_error_count(results_standardized, ground_truth)
number_no_vad = number_error_count(results_without_vad, ground_truth)
number_vad = number_error_count(results_conditional_vad, ground_truth)


# Construcción de tabla resumen
import pandas as pd

df_struct_errors = pd.DataFrame({
    "Pipeline": ["Standardized", "Sin VAD", "VAD condicional"],
    "Errores dominio": [domain_std, domain_no_vad, domain_vad],
    "Errores nombres": [name_std, name_no_vad, name_vad],
    "Errores números": [number_std, number_no_vad, number_vad]
})

# Cálculo del impacto total
df_struct_errors["Total impacto"] = (
    df_struct_errors["Errores dominio"] +
    df_struct_errors["Errores nombres"] +
    df_struct_errors["Errores números"]
)

display(df_struct_errors)

#### 7.2.5 Conclusiones

### 7.3 Impacto de los errores en etapas posteriores (PLN)

## 8. Comparativa de configuraciones de transcripción

### 8.1 Variación de parámetros del modelo
- baseline, el mejor que salio en el punto 6
- variante A: cambio un parametro (`beam_size`)
- variante B: cambio un parametro (`condition_on_previous_text`)
- variante C: combinación A + B

### 8.2 Evaluación comparativa (WER)
- comparo las 4 variantes anteriores con WER
- tiempo de ejecucion
- preguntar a GPT si mas métricas

### 8.3 Selección de configuración óptima

Elijo la mejor configuración justificando por qué.

## 9. Selección del *pipeline* final de transcripción

Al igual que en el notebook 1 crear aqui todo el pipeline bien estructurado con sus salidas.

## 10. Implementación y ejecución del *pipeline* final de ASR

### 10.1 Inicialización del entorno y rutas

### Implementación de funciones del *pipeline*

#### 10.2.1 Carga de audios

#### 10.2.2 Transcripción automática

#### 10.2.3 Postprocesado básico del texto

#### 10.2.4 Persistencia de resultados estructurados

### 10.3 Ejecución secuencial del *pipeline*

## 11. Conclusiones
- Resumen del proceso aplicado
- Resultados principales
- Relevancia para el pipeline de Speech-to-Text

In [ ]:
audio_id = "AUDIO-2025-09-18-13-34-03"

item = next(x for x in dataset_without_vad if x["audio_id"] == audio_id)

segments, _ = model.transcribe(str(item["path"]), **asr_params)

text = " ".join([s.text for s in segments]).strip()

print(text)